# Modelo global tipo panel para toda la flota

En este cuaderno queremos saber si un solo modelo entrenado con todos los inversores a la vez pronostica el rendimiento
del día siguiente tan bien como los modelos individuales de cada unidad, y si sirve para una planta que nunca vio. Los
modelos individuales aprenden la relación entre el tiempo pronosticado y el rendimiento de una sola unidad, con unos
cientos de días. Un modelo global (Montero-Manso y Hyndman, 2021; Januschowski et al., 2020) aprende esa relación con
todas las unidades juntas y distingue cada una por sus **covariables estáticas** (kWp, relación DC/AC, inclinación,
azimut, número de caras, ubicación, altitud, región y tipo), y por eso puede pronosticar una planta nueva desde el
primer día (**arranque en frío**), algo que un modelo individual no puede hacer.

Comparamos cuatro cosas. La primera es una **validación temporal** con las particiones de la flota, relativas al corte
del modelado, en la que la prueba son los dos meses anteriores al corte, la validación los dos previos y el
entrenamiento todo lo anterior dentro del régimen vigente. La segunda es una **validación agrupada** por localidad, que
deja fuera cada grupo meteorológico completo, y por planta, con pliegues que dejan fuera plantas completas con todos
sus inversores, para medir la generalización a instalaciones nuevas en lugares conocidos y desconocidos. La tercera es
el **arranque en frío**, en el que las unidades recientes, con menos de 14 días de entrenamiento, nunca entran al
entrenamiento y se pronostican con el modelo final. La cuarta es la comparación del **global frente al individual** con
Diebold-Mariano por unidad en las horas comunes de la prueba.

Los pronósticos usan solo información disponible al final del día anterior (escenario E2-MOS), que es el Yr del
pronóstico IFS 0,25° emitido el día anterior, la meteorología pronosticada, los rezagos de 24 h o más, la temperatura
del inversor de la misma hora del día anterior y la geometría solar.

In [ ]:
FIN = ""                       # fecha local de fin (exclusiva); "" = config.FIN_ANALISIS, la de los cuadernos por grupo
RAPIDO = False                 # True: prueba rápida (búsquedas y épocas mínimas)
GUARDAR_FIGURAS = True
SUF = ""                       # "_rapido": usa el panel de las ejecuciones rápidas de prueba

In [ ]:
import os, sys, warnings
from pathlib import Path
if not Path("tfm").is_dir() and Path("../tfm").is_dir():
    os.chdir("..")     # copia ejecutada abierta desde notebooks/ejecutados: se trabaja desde notebooks/
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown as MarkdownIPython
def Markdown(texto):                  # cifras con coma decimal, como en el documento y las figuras
    return MarkdownIPython(gr.texto_es(texto))
def miles(n):                         # separador de miles con espacio: 12 345 (la coma sería decimal)
    return f"{int(n):,}".replace(",", " ")
from sklearn.base import clone
from sklearn.model_selection import GroupKFold

from tfm import config, graficos as gr, resultados as res, metricas as met, particiones as part, tiempos
from tfm.modelos import ml

FIN = FIN or config.FIN_ANALISIS

tiempos.activar("GLOBAL")
config.fijar_semilla()
config.crear_directorios()
gr.estilo()
pd.set_option("display.max_columns", 40, "display.width", 200)
if RAPIDO:
    GUARDAR_FIGURAS = False
    res.registrar = lambda *a, **k: None
PREF = "global_"
DIAS_MIN_SARIMAX, DIAS_MIN_ML = 14, 60

def fig(f, nombre, datos=None):
    gr.guardar(f, PREF + nombre, GUARDAR_FIGURAS, datos)
    plt.show()
    if datos is not None:
        gr.mostrar(datos)
    plt.close(f)

## A. El panel

Cada cuaderno por grupo guardó, para cada inversor, su matriz del escenario E2 en las horas de sol, el
objetivo $Y_f$, la máscara de horas modelables (observadas, con como máximo una lectura de cinco minutos
imputada, sin falla ni recorte y dentro del régimen vigente) y sus covariables estáticas. Aquí se apilan.

In [ ]:
DIR_PANEL = config.DIR_INTERMEDIOS / f"panel{SUF}"
archivos = sorted(DIR_PANEL.glob("*.parquet"))
PANEL = pd.concat([pd.read_parquet(a) for a in archivos])
fin_utc = pd.Timestamp(FIN).tz_localize(config.ZONA).tz_convert("UTC")
PANEL = PANEL[PANEL.index <= fin_utc].sort_index(kind="stable")
# Unidades cuyo rendimiento no se puede normalizar: la potencia identificada con la telemetría es menos del 20 % de la
# registrada (r < 0,2) o, sin auditoría, su PR mediano es menor que 0,2. Su Yf no mide el proceso fotovoltaico (la
# potencia registrada no corresponde a lo que produce) y distorsionaria el entrenamiento y la evaluación de la flota
R_ = res.cargar().get("localidades", {})
EXCLUIDAS_KWP = {}
for g_, v_ in R_.items():
    aud_, obj_ = v_.get("datos", {}).get("auditoria_p0", {}), v_.get("objetivos", {})
    for u_ in set(aud_) | set(obj_):
        r_, pr_ = (aud_.get(u_) or {}).get("r"), (obj_.get(u_) or {}).get("pr_mediano")
        if (r_ is not None and r_ < 0.2) or (r_ is None and pr_ is not None and pr_ < 0.2):
            EXCLUIDAS_KWP[u_] = {"r": r_, "pr_mediano": pr_}
if EXCLUIDAS_KWP:
    PANEL = PANEL[~PANEL["unidad"].isin(list(EXCLUIDAS_KWP))]
    display(Markdown("**Fuera del panel por un kWp no verificable** (potencia identificada menor que el 20 % de la registrada o "
                     "PR mediano menor que 0,2): " + ", ".join(f"{u} (r = {v['r'] if v['r'] is not None else 'sin auditoría'}, "
                                                              f"PR mediano {v['pr_mediano']:.2f})" for u, v in sorted(EXCLUIDAS_KWP.items()))
                     + ". Su producción no corresponde a la potencia registrada, así que su Yf no mide el proceso fotovoltaico."))
PANEL["dia"] = part.dias_locales(PANEL.index)
ID = ["yf", "modelable", "unidad", "planta", "grupo", "particion", "frac_imputada", "dia"]
CATEGORICAS = ["region_natural", "tipo_instalacion"]
for c in CATEGORICAS:
    PANEL[c] = PANEL[c].fillna("sin dato").astype(str)
DUMMIES = pd.get_dummies(PANEL[CATEGORICAS], prefix=CATEGORICAS, dtype=float)
PANEL = pd.concat([PANEL.drop(columns=CATEGORICAS), DUMMIES], axis=1)

mod = PANEL[PANEL["modelable"]]
DIAS = mod.groupby(["unidad", "particion"])["dia"].nunique().unstack(fill_value=0)
for c in ("entrenamiento", "validacion", "prueba", "vivo"):
    if c not in DIAS:
        DIAS[c] = 0
DIAS = DIAS[["entrenamiento", "validacion", "prueba", "vivo"]]
DIAS["grupo"] = mod.groupby("unidad")["grupo"].first()
DIAS["planta"] = mod.groupby("unidad")["planta"].first()
DIAS["conjunto"] = np.where(DIAS["entrenamiento"] < DIAS_MIN_SARIMAX, "arranque en frío",
                            np.where(DIAS["entrenamiento"] < DIAS_MIN_ML, "conocida (pocos días)", "conocida"))
CONOCIDAS = DIAS.index[DIAS["conjunto"] != "arranque en frío"].tolist()
FRIO = DIAS.index[DIAS["conjunto"] == "arranque en frío"].tolist()
display(Markdown(f"**{PANEL['unidad'].nunique()} inversores en {PANEL['grupo'].nunique()} grupos**, {miles(len(PANEL))} horas de sol, "
                 f"{miles(PANEL['modelable'].sum())} modelables. Días modelables por unidad y partición:"))
display(DIAS.sort_values(["grupo", "planta"]))
display(Markdown(f"**Arranque en frío** (menos de {DIAS_MIN_SARIMAX} días de entrenamiento; nunca se entrenan): "
                 + (", ".join(FRIO) if FRIO else "ninguna") + "."))
res.registrar("global.panel", {"inversores": int(PANEL["unidad"].nunique()), "grupos": int(PANEL["grupo"].nunique()),
              "excluidas_kwp": sorted(EXCLUIDAS_KWP),
              "horas_sol": int(len(PANEL)), "horas_modelables": int(PANEL["modelable"].sum()),
              "dias_por_unidad": DIAS.drop(columns=["grupo", "planta"]).to_dict(orient="index"), "arranque_en_frio": FRIO})

### A.1 Variables

Las mismas candidatas del escenario E2 de los cuadernos por grupo, más la temperatura del inversor con 24 h
de rezago y las covariables estáticas. Se descartan las variables con más del 40 % de faltantes en el
entrenamiento (por ejemplo, las que solo existen en algunos grupos). Los faltantes restantes se imputan con
la mediana del entrenamiento dentro del *pipeline* (bosque aleatorio) o se tratan de forma nativa (HGB y
LightGBM).

In [ ]:
TR = (PANEL["modelable"] & (PANEL["particion"] == "entrenamiento") & PANEL["unidad"].isin(CONOCIDAS)).to_numpy()
VA = (PANEL["modelable"] & (PANEL["particion"] == "validacion") & PANEL["unidad"].isin(CONOCIDAS)).to_numpy()
PR = (PANEL["modelable"] & (PANEL["particion"] == "prueba") & PANEL["unidad"].isin(CONOCIDAS)).to_numpy()
# Arranque en frío: todas las horas modelables de las unidades recientes hasta el corte del modelado (las del periodo en
# vivo, posteriores al corte, se evalúan aparte)
FR = (PANEL["modelable"] & PANEL["unidad"].isin(FRIO) & (PANEL["particion"] != "vivo")).to_numpy()
TRVA = TR | VA
candidatas = [c for c in PANEL.columns if c not in ID]
faltantes = PANEL.loc[TR, candidatas].isna().mean()
VARS = [c for c in candidatas if faltantes[c] <= 0.40 and PANEL.loc[TR, c].nunique() > 1]
ESTATICAS = [c for c in VARS if c in ("kwp", "potencia_nominal_kw", "relacion_dc_ac", "inclinacion_media", "azimut_sen",
                                      "azimut_cos", "caras", "latitud", "longitud", "altitud_m") or c.startswith(tuple(CATEGORICAS))]
display(Markdown(f"**{len(VARS)} variables** ({len(ESTATICAS)} estáticas): " + ", ".join(f"`{c}`" for c in VARS)))
descartadas = sorted(set(candidatas) - set(VARS))
if descartadas:
    display(Markdown("Descartadas (más del 40 % de faltantes o constantes): " + ", ".join(f"`{c}`" for c in descartadas)))
X, y = PANEL[VARS], PANEL["yf"]
UNIDAD_FILA = PANEL["unidad"].to_numpy()
# Referencias por fila, calculables para cualquier unidad desde su segundo día de datos
REF = {"ref: persistencia estacional": PANEL["yf_rez24"].to_numpy(),
       "ref: persistencia inteligente": (PANEL["kstar_rez24"] * PANEL["yr_cs"]).to_numpy()}
# CLIPER por unidad, la misma referencia de los modelos individuales: el k* de la misma hora del día anterior mezclado
# con su media del entrenamiento según su autocorrelación a 24 h. Las unidades sin entrenamiento (arranque en frío)
# usan la autocorrelación y la media de la flota.
KST = (PANEL["yf"] / PANEL["yr_cs"]).where(PANEL["yr_cs"] > 0.05)
POS_UNIDAD = PANEL.groupby("unidad").indices
def _horaria(u):
    g = PANEL.iloc[POS_UNIDAD[u]]
    H_ = pd.DataFrame({"k": KST.iloc[POS_UNIDAD[u]].to_numpy(), "cs": g["yr_cs"].to_numpy(),
                       "tr": TR[POS_UNIDAD[u]]}, index=g.index)
    return H_[~H_.index.duplicated()].asfreq("1h")
RHO_U = {}
for u in POS_UNIDAD:
    H_ = _horaria(u)
    k_tr = H_["k"].where(H_["tr"].fillna(False).astype(bool))
    if k_tr.notna().sum() > 48:
        RHO_U[u] = (float(np.clip(k_tr.autocorr(24), 0, 1)), float(k_tr.mean()))
RHO_FLOTA = float(np.median([v[0] for v in RHO_U.values()])) if RHO_U else 0.0
K_MEDIA_FLOTA = float(KST[TR].mean())
cl_ = np.full(len(PANEL), np.nan)
for u, pos in POS_UNIDAD.items():
    H_ = _horaria(u)
    rho_, km_ = RHO_U.get(u, (RHO_FLOTA, K_MEDIA_FLOTA))
    c_ = (rho_ * H_["k"].shift(24).fillna(km_) + (1 - rho_) * km_) * H_["cs"]
    cl_[pos] = c_.reindex(PANEL.index[pos]).to_numpy()
REF["ref: cliper"] = cl_
display(pd.Series({"entrenamiento": int(TR.sum()), "validación": int(VA.sum()), "prueba": int(PR.sum()),
                   "arranque en frío": int(FR.sum())}, name="horas modelables").to_frame())

## B. Validación temporal: búsqueda de hiperparámetros y validación

Con el mismo proceso de los modelos individuales: búsqueda aleatoria y luego en cuadrícula con validación
cruzada temporal de 4 pliegues sin barajar (las filas están ordenadas en el tiempo, así que cada pliegue
valida con horas posteriores a las de su ajuste). Para que la búsqueda sea manejable con cientos de miles de
filas se hace sobre una muestra aleatoria de 80 000 horas del entrenamiento que conserva el orden temporal;
el mejor modelo se reentrena con todo el entrenamiento y se evalúa en la validación. El ensamble promedia
los tres modelos de árboles.

In [ ]:
MODELOS_G = ["RF", "HGB", "LGBM"] if not RAPIDO else ["LGBM"]
rng = np.random.default_rng(config.SEMILLA)
idx_tr = np.flatnonzero(TR)
muestra = np.sort(rng.choice(idx_tr, size=min(80_000, len(idx_tr)), replace=False))
AJ, PRED_VA, BUSQ = {}, {}, []
for m in MODELOS_G:
    with tiempos.medir(f"busqueda {m}", "busqueda", "global"):
        aj = ml.ajustar(m, X.iloc[muestra], y.iloc[muestra], n_iter=12 if not RAPIDO else 2, pliegues=4, rejilla=not RAPIDO)
    mejor = clone(aj["mejor"])
    if m == "RF":
        mejor.set_params(modelo__n_jobs=config.N_JOBS)
    with tiempos.medir(f"ajuste {m}", "ajuste_final", "global"):
        mejor.fit(X[TR], y[TR])
    AJ[m] = mejor
    PRED_VA[m] = np.clip(mejor.predict(X[VA]), 0, None)
    BUSQ.append({"modelo": m, "mae_cv": aj.get("mae_cv_rejilla", aj["mae_cv_aleatoria"]), "segundos_busqueda": aj["segundos"],
                 "parametros": str(aj["parametros"])})
if len(MODELOS_G) > 1:
    PRED_VA["ENS"] = np.mean([PRED_VA[m] for m in MODELOS_G], axis=0)
display(pd.DataFrame(BUSQ).set_index("modelo"))
res.registrar("global.busqueda", pd.DataFrame(BUSQ).set_index("modelo").to_dict(orient="index"))

### B.1 Redes neuronales

**MLP** sobre las mismas variables, escaladas con la media y la desviación del entrenamiento (los faltantes,
con la mediana del entrenamiento). **LSTM** bidireccional que pronostica a la vez las 14 horas de 05 a 19 h
de un día de una unidad a partir de la secuencia de sus entradas de ese día, todas conocidas al emitir el
pronóstico; las horas no modelables entran como ceros y pesan 0 en la pérdida. Parada temprana con el último
15 % del entrenamiento en el tiempo, como en los cuadernos por grupo.

In [ ]:
import keras
from tfm.modelos import redes
redes.configurar()
mediana = X[TR].median()
media, desv = X[TR].fillna(mediana).mean(), X[TR].fillna(mediana).std().replace(0, 1)
escalar = lambda A: ((A.fillna(mediana) - media) / desv).to_numpy(dtype="float32")
EPOCAS = 200 if not RAPIDO else 3

def ajustar_mlp(mascara):
    m = redes.mlp(len(VARS), capas=(128, 64))
    redes.entrenar(m, escalar(X[mascara]), y[mascara].to_numpy(dtype="float32"), epocas=EPOCAS, paciencia=15, lote=256)
    return m

HORAS = np.arange(5, 19)
HORA_FILA = np.asarray(((PANEL.index - pd.Timedelta("30min")).tz_convert(config.ZONA)).hour)
DIA_FILA = PANEL["dia"].to_numpy()

def tensor(pos):
    # Filas `pos` de PANEL -> (unidad-día, 14 horas, variables), en orden temporal. Devuelve también la
    # posición (día, hora) de cada fila usada para volver a las filas
    pos = pos[np.isin(HORA_FILA[pos], HORAS)]
    clave = pd.MultiIndex.from_arrays([DIA_FILA[pos], UNIDAD_FILA[pos]])
    claves = clave.unique().sort_values()
    pd_, ph_ = claves.get_indexer(clave), np.searchsorted(HORAS, HORA_FILA[pos])
    Xa = np.zeros((len(claves), len(HORAS), len(VARS)), dtype="float32")
    ya = np.zeros((len(claves), len(HORAS)), dtype="float32")
    wa = np.zeros((len(claves), len(HORAS)), dtype="float32")
    Xa[pd_, ph_] = escalar(X.iloc[pos])
    ok = PANEL["modelable"].to_numpy()[pos] & np.isfinite(y.to_numpy()[pos])
    ya[pd_[ok], ph_[ok]] = y.to_numpy(dtype="float32")[pos][ok]
    wa[pd_[ok], ph_[ok]] = 1.0
    return Xa, ya, wa, pos, pd_, ph_

def filas_contexto(mascara):
    # Todas las horas de sol (modelables o no) de las unidades y días de la máscara: entradas completas
    u, d = np.unique(UNIDAD_FILA[mascara]), np.unique(DIA_FILA[mascara])
    return np.flatnonzero(np.isin(UNIDAD_FILA, u) & np.isin(DIA_FILA, d))

def lstm_panel(n_entradas, unidades=48):
    entrada = keras.Input(shape=(len(HORAS), n_entradas))
    x = keras.layers.Bidirectional(keras.layers.LSTM(unidades, return_sequences=True))(entrada)
    x = keras.layers.Dropout(0.2)(x)
    x = keras.layers.TimeDistributed(keras.layers.Dense(16, activation="relu"))(x)
    salida = keras.layers.TimeDistributed(keras.layers.Dense(1, activation="relu"))(x)
    m = keras.Model(entrada, salida, name="LSTM_global")
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mae")
    return m

def ajustar_lstm(mascara):
    Xa, ya, wa, *_ = tensor(filas_contexto(mascara))
    # Solo cuentan en la pérdida las horas de la máscara (no las de contexto de otras particiones)
    m = lstm_panel(len(VARS))
    corte = int(len(Xa) * 0.85)
    parada = keras.callbacks.EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True)
    m.fit(Xa[:corte], ya[:corte, :, None], sample_weight=wa[:corte],
          validation_data=(Xa[corte:], ya[corte:, :, None], wa[corte:]), epochs=EPOCAS, batch_size=64,
          callbacks=[parada], verbose=0, shuffle=True)
    return m

def predecir_lstm(m, mascara):
    Xa, _, _, pos, pd_, ph_ = tensor(filas_contexto(mascara))
    p = m.predict(Xa, verbose=0)[:, :, 0].clip(0)
    completa = np.full(len(PANEL), np.nan)
    completa[pos] = p[pd_, ph_]
    return completa[mascara]

with tiempos.medir("ajuste MLP", "ajuste_final", "global"):
    MLP = ajustar_mlp(TR)
PRED_VA["MLP"] = MLP.predict(escalar(X[VA]), verbose=0).ravel().clip(0)
# Para la LSTM la máscara de entrenamiento es por partición: las horas de contexto de la validación no se usan
with tiempos.medir("ajuste LSTM", "ajuste_final", "global"):
    LSTM = ajustar_lstm(TR)
PRED_VA["LSTM"] = predecir_lstm(LSTM, VA)

### B.2 Resultados en la validación

Error de cada modelo en la validación, por unidad y para toda la flota, con las referencias de persistencia
estacional (misma hora del día anterior) y persistencia inteligente (el índice de cielo despejado de la misma
hora del día anterior por el $Y_r$ de cielo despejado). Todas las cifras de una tabla usan las mismas horas:
las que tienen el valor observado y la predicción de todos los modelos y referencias.

In [ ]:
def comparar(preds, mascara):
    yy, uu = y.to_numpy()[mascara], UNIDAD_FILA[mascara]
    todos = {**preds, **{k: v[mascara] for k, v in REF.items()}}
    comun = np.isfinite(yy) & np.all([np.isfinite(p) for p in todos.values()], axis=0)
    flota = pd.DataFrame({k: met.evaluar(yy[comun], p[comun]) for k, p in todos.items()}).T
    for c in ("mae", "rmse", "mbe", "nmae_pct", "nrmse_pct", "r2"):
        if c not in flota:
            flota[c] = np.nan
    flota["skill_vs_persistencia_inteligente"] = 1 - flota["rmse"] / flota.loc["ref: persistencia inteligente", "rmse"]
    flota["skill_vs_cliper"] = 1 - flota["rmse"] / flota.loc["ref: cliper", "rmse"]
    filas = [{"modelo": k, "unidad": u, **met.evaluar(yy[comun & (uu == u)], p[comun & (uu == u)])}
             for k, p in todos.items() for u in np.unique(uu[comun])]
    por_unidad = pd.DataFrame(filas) if filas else pd.DataFrame(columns=["modelo", "unidad", "n", "nrmse_pct"])
    return flota.sort_values("rmse"), por_unidad

FLOTA_VA, UNI_VA = comparar(PRED_VA, VA)
display(Markdown("**Validación, toda la flota**")); display(FLOTA_VA.round(4))
display(UNI_VA.pivot(index="unidad", columns="modelo", values="nrmse_pct").round(2))
MEJOR_G = FLOTA_VA.index[~FLOTA_VA.index.str.startswith("ref")][0]
display(Markdown(f"**Modelo global elegido en la validación: {MEJOR_G}** (menor RMSE de la flota)."))
res.registrar("global.validacion", {"flota": FLOTA_VA.round(5).to_dict(orient="index"), "elegido": MEJOR_G,
              "nrmse_por_unidad": UNI_VA.pivot(index="unidad", columns="modelo", values="nrmse_pct").round(3).to_dict(orient="index")})

## C. Prueba temporal y comparación con los modelos individuales

Los modelos se reentrenan con el entrenamiento y la validación, y se evalúan en la prueba (los dos meses
anteriores al corte del modelado). Para cada unidad conocida se compara el modelo global elegido en la validación
con el modelo individual elegido en su cuaderno (`modelo_final`) y con el mejor modelo individual de la prueba, en
las horas comunes, con Diebold-Mariano sobre la pérdida cuadrática sumada por día (los errores de las horas de un
mismo día vienen del mismo pronóstico) y la corrección de Holm entre unidades, porque es la misma pregunta en
decenas de unidades.

In [ ]:
PRED_PR = {}
for m in MODELOS_G:
    with tiempos.medir(f"ajuste final {m}", "ajuste_final", "global"):
        AJ[m] = clone(AJ[m]).fit(X[TRVA], y[TRVA])
    PRED_PR[m] = np.clip(AJ[m].predict(X[PR]), 0, None)
if len(MODELOS_G) > 1:
    PRED_PR["ENS"] = np.mean([PRED_PR[m] for m in MODELOS_G], axis=0)
MLP = ajustar_mlp(TRVA)
PRED_PR["MLP"] = MLP.predict(escalar(X[PR]), verbose=0).ravel().clip(0)
LSTM = ajustar_lstm(TRVA)
PRED_PR["LSTM"] = predecir_lstm(LSTM, PR)
FLOTA_PR, UNI_PR = comparar(PRED_PR, PR)
display(Markdown("**Prueba, toda la flota**")); display(FLOTA_PR.round(4))
display(UNI_PR.pivot(index="unidad", columns="modelo", values="nrmse_pct").round(2))
res.registrar("global.prueba", {"flota": FLOTA_PR.round(5).to_dict(orient="index"),
              "nrmse_por_unidad": UNI_PR.pivot(index="unidad", columns="modelo", values="nrmse_pct").round(3).to_dict(orient="index")})

In [ ]:
DIR_PRED = config.DIR_INTERMEDIOS / f"predicciones{SUF}"
RESULTADOS = res.cargar()
filas = []
idx_pr = PANEL.index[PR]
for u in CONOCIDAS:
    g = DIAS.loc[u, "grupo"]
    archivo = DIR_PRED / f"{g}_{u}_prueba.parquet"
    if not archivo.exists():
        continue
    loc = pd.read_parquet(archivo)
    su = UNIDAD_FILA[PR] == u
    pg = pd.Series(PRED_PR[MEJOR_G][su], index=idx_pr[su])
    comun = loc.index.intersection(pg.index)
    comun = comun[loc.loc[comun, "y"].notna().to_numpy() & pg.loc[comun].notna().to_numpy()]
    if len(comun) < 48:
        continue
    yy = loc.loc[comun, "y"]
    elegido = RESULTADOS.get("localidades", {}).get(g, {}).get("unidades", {}).get(u, {}).get("modelo_final", {}).get("modelo")
    modelos_l = [c for c in loc.columns if c != "y" and not c.startswith("ref") and loc.loc[comun, c].notna().all()]
    if not modelos_l:
        continue
    mejor_l = min(modelos_l, key=lambda c: met.evaluar(yy, loc.loc[comun, c])["rmse"])
    elegido = elegido if elegido in modelos_l else mejor_l
    e_g, e_e, e_m = (met.evaluar(yy, s) for s in (pg.loc[comun], loc.loc[comun, elegido], loc.loc[comun, mejor_l]))
    dm_e = met.diebold_mariano_diario(yy, pg.loc[comun], loc.loc[comun, elegido])
    dm_m = met.diebold_mariano_diario(yy, pg.loc[comun], loc.loc[comun, mejor_l])
    cliper = met.evaluar(yy, loc.loc[comun, "ref: cliper"])["rmse"] if "ref: cliper" in loc else np.nan
    dm_c = met.diebold_mariano_diario(yy, pg.loc[comun], loc.loc[comun, "ref: cliper"]) if "ref: cliper" in loc else {}
    filas.append({"unidad": u, "grupo": g, "horas": len(comun), "nrmse_global": e_g["nrmse_pct"],
                  "modelo_individual": elegido, "nrmse_individual": e_e["nrmse_pct"], "dm_vs_individual": dm_e.get("dm"),
                  "p_vs_individual": dm_e.get("p_valor"), "mejor_individual_prueba": mejor_l, "nrmse_mejor_individual": e_m["nrmse_pct"],
                  "p_vs_mejor_individual": dm_m.get("p_valor"),
                  "skill_global_vs_cliper": 1 - e_g["rmse"] / cliper if cliper else np.nan,
                  "skill_individual_vs_cliper": 1 - e_e["rmse"] / cliper if cliper else np.nan,
                  "dm_vs_cliper": dm_c.get("dm"), "p_vs_cliper": dm_c.get("p_valor")})
GLOBAL_VS_LOCAL = pd.DataFrame(filas).set_index("unidad") if filas else pd.DataFrame()
if not GLOBAL_VS_LOCAL.empty:
    GLOBAL_VS_LOCAL["p_vs_individual_holm"] = met.holm(GLOBAL_VS_LOCAL["p_vs_individual"])
    GLOBAL_VS_LOCAL["veredicto"] = np.select(
        [(GLOBAL_VS_LOCAL["p_vs_individual_holm"] < 0.05) & (GLOBAL_VS_LOCAL["dm_vs_individual"] < 0),
         (GLOBAL_VS_LOCAL["p_vs_individual_holm"] < 0.05) & (GLOBAL_VS_LOCAL["dm_vs_individual"] > 0)],
        ["global mejor", "individual mejor"], "sin diferencia significativa")
    display(Markdown(f"**{MEJOR_G} global frente al modelo individual elegido en cada unidad (prueba)**"))
    display(GLOBAL_VS_LOCAL.round(4))
    conteo = GLOBAL_VS_LOCAL["veredicto"].value_counts()
    display(Markdown("En la comparación por unidad, " + "; ".join(f"{k}: {v} unidades" for k, v in conteo.items()) +
                     f". nRMSE mediano: global {GLOBAL_VS_LOCAL['nrmse_global'].median():.1f} %, individual "
                     f"{GLOBAL_VS_LOCAL['nrmse_individual'].median():.1f} %."))
    res.registrar("global.vs_individual", {"modelo_global": MEJOR_G, "tabla": GLOBAL_VS_LOCAL.round(5).to_dict(orient="index"),
                  "conteo": conteo.to_dict()})
    t = GLOBAL_VS_LOCAL.sort_values("nrmse_individual")
    t = t.assign(diferencia=t["nrmse_global"] - t["nrmse_individual"])
    # Dos paneles en la misma fila con el eje de unidades compartido: a la izquierda los dos errores (puntos), a la
    # derecha su diferencia (barras; negativa = el global es mejor). Naranja: el global es mejor con significancia;
    # gris oscuro: el individual es mejor con significancia; gris claro: sin diferencia significativa.
    f, (ax, axd) = plt.subplots(1, 2, figsize=(12, 2 * (0.15 * len(t) + 1.4)), sharey=True,
                                gridspec_kw={"width_ratios": [3, 2], "wspace": 0.04})
    yy_ = np.arange(len(t))
    ax.hlines(yy_, t[["nrmse_global", "nrmse_individual"]].min(axis=1), t[["nrmse_global", "nrmse_individual"]].max(axis=1),
              color=config.COLOR_NEUTRO, lw=0.8)
    ax.scatter(t["nrmse_individual"], yy_, color=config.COLOR_SECUNDARIO, s=12, label="modelo individual elegido", zorder=3)
    ax.scatter(t["nrmse_global"], yy_, color=config.COLOR_PRINCIPAL, s=12, marker="D", label=f"modelo global ({MEJOR_G})", zorder=3)
    for i, (u, r) in enumerate(t.iterrows()):
        if r["veredicto"] != "sin diferencia significativa":
            ax.annotate("*", (max(r["nrmse_global"], r["nrmse_individual"]) + 0.4, i), va="center")
    ax.set_yticks(yy_, [f"{u} ({r['grupo']})" for u, r in t.iterrows()])
    ax.set_ylim(len(t) - 0.5, -0.5)
    ax.set_xlim(left=0); ax.set_xlabel("nRMSE horario en la prueba (%)")
    # Abajo a la izquierda: los errores bajos de las últimas filas dejan ahí el mayor espacio libre
    ax.legend(loc="lower left", markerscale=1.5)
    color_d = np.where(t["veredicto"] == "global mejor", config.COLOR_PRINCIPAL,
                       np.where(t["veredicto"] == "individual mejor", "#4D4D4D", "#BDBDBD"))
    axd.barh(yy_, t["diferencia"], color=color_d, height=0.7)
    axd.axvline(0, color="black", lw=0.8)
    axd.set_xlabel("Global menos individual (puntos de nRMSE)")
    axd.tick_params(axis="y", left=False)
    f.suptitle("Modelo global frente al individual por unidad (*: Diebold-Mariano diario con Holm, p < 0,05)")
    fig(f, "global_vs_individual", t)

### C.1 Validación por bloques del modelo global

Para contrastar la comparación con los modelos individuales en más de una prueba, el modelo global se reajusta en
las mismas fechas que la validación por bloques de las unidades (1 de noviembre, febrero, mayo y agosto), con los
datos observados de las unidades conocidas antes de cada fecha, y pronostica cada día del trimestre. Los
hiperparámetros son los elegidos en la validación (B); en los bloques anteriores a la validación eso da una pequeña
ventaja de información sobre los hiperparámetros, no sobre los coeficientes, que se declara. Las predicciones de
cada unidad quedan en `resultados/intermedios/bloques_global/` y se comparan en `05_bloques.ipynb`.

In [ ]:
REAJUSTES_G = [pd.Timestamp(d) for d in ("2025-11-01", "2026-02-01", "2026-05-01", "2026-08-01")]
DIR_BG = config.DIR_INTERMEDIOS / f"bloques_global{SUF}"
DIR_BG.mkdir(parents=True, exist_ok=True)
dias_p = PANEL["dia"].to_numpy()
pred_b = np.full(len(PANEL), np.nan)
modelos_b = [MEJOR_G] if MEJOR_G in MODELOS_G else MODELOS_G      # el ensamble se reconstruye con sus componentes
for i, r in enumerate(REAJUSTES_G):
    fin_r = REAJUSTES_G[i + 1] if i + 1 < len(REAJUSTES_G) else pd.Timestamp(FIN)
    tr_b = (PANEL["modelable"].to_numpy() & PANEL["unidad"].isin(CONOCIDAS).to_numpy() & (dias_p < r))
    en_b = (dias_p >= r) & (dias_p < fin_r)
    if tr_b.sum() < 1000 or not en_b.any():
        continue
    with tiempos.medir(f"bloque {r:%Y-%m}", "ajuste_bloques", "global"):
        preds = [np.clip(clone(AJ[m]).fit(X[tr_b], y[tr_b]).predict(X[en_b]), 0, None) for m in modelos_b]
    pred_b[en_b] = np.mean(preds, axis=0)
for u, pos in POS_UNIDAD.items():
    s = pd.Series(pred_b[pos], index=PANEL.index[pos], name="global").dropna()
    if len(s):
        s.to_frame().to_parquet(DIR_BG / f"{u}.parquet")
display(Markdown(f"Predicciones por bloques del modelo global ({MEJOR_G}) guardadas para {len(list(DIR_BG.glob('*.parquet')))} unidades."))

## D. Validación agrupada: plantas y localidades que el modelo no vio

Para cada grupo meteorológico entrenamos los modelos de árboles con el entrenamiento y la validación de las unidades
conocidas de los demás grupos, con los hiperparámetros elegidos en B, y los evaluamos en la prueba de las unidades del
grupo excluido. Hacemos lo mismo por planta, con 5 pliegues que dejan fuera plantas completas con todos sus inversores
aunque su grupo sí esté en el entrenamiento. La comparación con la prueba temporal (C), en la que cada unidad sí estaba
en el entrenamiento, mide cuánto se pierde al pronosticar una instalación nueva. No repetimos aquí las redes por su
costo y su inestabilidad.

In [ ]:
def agrupada(etiqueta, divisiones):
    etq = PANEL[etiqueta].to_numpy()
    preds = {m: np.full(int(PR.sum()), np.nan) for m in MODELOS_G}
    etq_pr = etq[PR]
    for fuera in divisiones:
        entren = TRVA & ~np.isin(etq, fuera)
        sel = np.isin(etq_pr, fuera)
        if not sel.any() or entren.sum() < 1000:
            continue
        for m in MODELOS_G:
            mod_k = clone(AJ[m]).fit(X[entren], y[entren])
            preds[m][sel] = np.clip(mod_k.predict(X[PR][sel]), 0, None)
    if len(MODELOS_G) > 1:
        preds["ENS"] = np.mean([preds[m] for m in MODELOS_G], axis=0)
    return preds

with tiempos.medir("agrupada por localidad", "validacion_agrupada", "global"):
    PRED_LOGO = agrupada("grupo", [[g] for g in sorted(np.unique(PANEL["grupo"].to_numpy()[PR]))])
plantas_c = np.array(sorted(np.unique(PANEL["planta"].to_numpy()[TRVA | PR])))
with tiempos.medir("agrupada por planta", "validacion_agrupada", "global"):
    PRED_PLANTA = agrupada("planta", [list(plantas_c[i::5]) for i in range(5)])
m_ref = MEJOR_G if MEJOR_G in PRED_LOGO else ("ENS" if "ENS" in PRED_LOGO else MODELOS_G[-1])
_, u_t = comparar({m_ref: PRED_PR[m_ref]}, PR)
_, u_l = comparar({m_ref: PRED_LOGO[m_ref]}, PR)
_, u_p = comparar({m_ref: PRED_PLANTA[m_ref]}, PR)
sel_ = lambda t: t[t["modelo"] == m_ref].set_index("unidad")["nrmse_pct"]
COMP_AGR = pd.DataFrame({"unidad conocida": sel_(u_t), "planta nueva": sel_(u_p), "localidad nueva": sel_(u_l)})
COMP_AGR["grupo"] = DIAS["grupo"]
display(Markdown(f"**nRMSE (%) en la prueba de {m_ref} según lo que el modelo vio de cada unidad**"))
display(COMP_AGR.round(2))
FLOTA_AGR = pd.DataFrame({"unidad conocida": comparar(PRED_PR, PR)[0]["nrmse_pct"],
                          "planta nueva": comparar(PRED_PLANTA, PR)[0]["nrmse_pct"],
                          "localidad nueva": comparar(PRED_LOGO, PR)[0]["nrmse_pct"]})
display(FLOTA_AGR.round(2))
res.registrar("global.agrupada", {"modelo": m_ref, "por_unidad": COMP_AGR.drop(columns="grupo").round(3).to_dict(orient="index"),
              "flota": FLOTA_AGR.round(3).to_dict(orient="index")})

# Misma muestra para la figura de cajas de la flota (cuaderno de comparación): en cada unidad conocida, las horas comunes
# de la prueba de su cuaderno y del panel, con CLIPER, el SARIMAX E2-MOS individual, el global con la unidad en el
# entrenamiento y el global con su planta fuera (lo que tendría una planta nueva)
filas_cj = []
for u in CONOCIDAS:
    archivo = DIR_PRED / f"{DIAS.loc[u, 'grupo']}_{u}_prueba.parquet"
    if not archivo.exists():
        continue
    loc = pd.read_parquet(archivo)
    if "SARIMAX E2-MOS" not in loc or "ref: cliper" not in loc:
        continue
    su = UNIDAD_FILA[PR] == u
    g_con = pd.Series(PRED_PR[m_ref][su], index=idx_pr[su])
    g_nue = pd.Series(PRED_PLANTA[m_ref][su], index=idx_pr[su])
    comun = loc.index.intersection(g_con.index)
    ok = (loc.loc[comun, ["y", "SARIMAX E2-MOS", "ref: cliper"]].notna().all(axis=1).to_numpy()
          & g_con.reindex(comun).notna().to_numpy() & g_nue.reindex(comun).notna().to_numpy())
    comun = comun[ok]
    if len(comun) < 48:
        continue
    yy = loc.loc[comun, "y"]
    filas_cj.append({"unidad": u, "grupo": DIAS.loc[u, "grupo"], "horas": len(comun),
                     "cliper": met.evaluar(yy, loc.loc[comun, "ref: cliper"])["nrmse_pct"],
                     "sarimax_individual": met.evaluar(yy, loc.loc[comun, "SARIMAX E2-MOS"])["nrmse_pct"],
                     "global_conocida": met.evaluar(yy, g_con.loc[comun])["nrmse_pct"],
                     "global_planta_nueva": met.evaluar(yy, g_nue.loc[comun])["nrmse_pct"]})
CAJAS = pd.DataFrame(filas_cj).set_index("unidad") if filas_cj else pd.DataFrame()
if len(CAJAS):
    display(Markdown(f"**Misma muestra por unidad para la figura de cajas de la flota** ({len(CAJAS)} unidades; modelo global {m_ref})"))
    display(CAJAS.round(2))
    res.registrar("global.cajas_flota", {"modelo": m_ref, "por_unidad": CAJAS.drop(columns="grupo").round(3).to_dict(orient="index")})

## E. Arranque en frío: las unidades recientes

Las unidades con menos de 14 días de entrenamiento no tienen historia suficiente para un modelo individual
en las particiones fijas. El modelo global las pronostica sin haberlas visto, solo con sus covariables
estáticas, el pronóstico meteorológico y sus propios rezagos (que existen desde su segundo día de datos). Se
evalúan todas sus horas modelables hasta el corte del modelado (`config.FIN_MODELADO`).

In [ ]:
if FR.any():
    PRED_FR = {m: np.clip(AJ[m].predict(X[FR]), 0, None) for m in MODELOS_G}
    if len(MODELOS_G) > 1:
        PRED_FR["ENS"] = np.mean([PRED_FR[m] for m in MODELOS_G], axis=0)
    PRED_FR["MLP"] = MLP.predict(escalar(X[FR]), verbose=0).ravel().clip(0)
    PRED_FR["LSTM"] = predecir_lstm(LSTM, FR)
    FLOTA_FR, UNI_FR = comparar(PRED_FR, FR)
    TAB_FR = UNI_FR.pivot(index="unidad", columns="modelo", values="nrmse_pct")
    TAB_FR["horas"] = UNI_FR.groupby("unidad")["n"].max()
    display(Markdown("**Arranque en frío: nRMSE (%) por unidad reciente**")); display(TAB_FR.round(2))
    display(FLOTA_FR.round(4))
    res.registrar("global.arranque_en_frio", {"nrmse_por_unidad": TAB_FR.round(3).to_dict(orient="index"),
                  "flota": FLOTA_FR.round(5).to_dict(orient="index")})
else:
    display(Markdown("No hay unidades de arranque en frío con horas modelables hasta el corte del modelado."))

## F. Importancia de las variables

Importancia por permutación del modelo de árboles final en la prueba (aumento del MAE al barajar cada
variable): muestra si las covariables estáticas aportan, es decir, si el modelo distingue las unidades.

In [ ]:
from sklearn.inspection import permutation_importance
m_imp = "LGBM" if "LGBM" in AJ else MODELOS_G[0]
idx_imp = np.flatnonzero(PR)
idx_imp = np.sort(rng.choice(idx_imp, size=min(30_000, len(idx_imp)), replace=False))
imp_ = permutation_importance(AJ[m_imp], X.iloc[idx_imp], y.iloc[idx_imp], scoring="neg_mean_absolute_error",
                              n_repeats=3 if not RAPIDO else 1, random_state=config.SEMILLA, n_jobs=config.N_JOBS)
IMP = pd.DataFrame({"aumento_mae": imp_.importances_mean, "desv": imp_.importances_std}, index=VARS).sort_values("aumento_mae", ascending=False)
IMP["estatica"] = IMP.index.isin(ESTATICAS)
t = IMP.head(20).iloc[::-1]
f, ax = plt.subplots(figsize=(9, 5.5))
ax.barh(t.index, t["aumento_mae"].clip(lower=0), xerr=t["desv"],
        color=[config.COLOR_SECUNDARIO if e else config.COLOR_PRINCIPAL for e in t["estatica"]])
ax.set_xlabel("Aumento del MAE al permutar (kWh/kWp)"); ax.set_xlim(left=0)
ax.set_title(f"Importancia por permutación, {m_imp} global (azul: covariable estática)")
fig(f, "importancia_permutacion", IMP)
res.registrar("global.importancia", {"modelo": m_imp, "top": IMP.head(20)["aumento_mae"].round(6).to_dict(),
              "aporte_estaticas": float(IMP.loc[IMP["estatica"], "aumento_mae"].clip(lower=0).sum()
                                        / max(IMP["aumento_mae"].clip(lower=0).sum(), 1e-12))})

## G. Conclusiones

In [ ]:
lineas = [f"- Modelo global elegido en la validación: **{MEJOR_G}**, nRMSE de la flota en la prueba "
          f"{FLOTA_PR.loc[MEJOR_G, 'nrmse_pct']:.1f} % (CLIPER {FLOTA_PR.loc['ref: cliper', 'nrmse_pct']:.1f} %, skill "
          f"{100 * FLOTA_PR.loc[MEJOR_G, 'skill_vs_cliper']:.1f} %; persistencia inteligente "
          f"{FLOTA_PR.loc['ref: persistencia inteligente', 'nrmse_pct']:.1f} %)."]
if not GLOBAL_VS_LOCAL.empty:
    c = GLOBAL_VS_LOCAL["veredicto"].value_counts()
    lineas.append("- Frente al modelo individual elegido en cada unidad: " + ", ".join(f"{k} en {v}" for k, v in c.items()) + ".")
lineas.append(f"- {m_ref}: unidad conocida {FLOTA_AGR.loc[m_ref, 'unidad conocida']:.1f} %, planta nueva "
              f"{FLOTA_AGR.loc[m_ref, 'planta nueva']:.1f} %, localidad nueva {FLOTA_AGR.loc[m_ref, 'localidad nueva']:.1f} %.")
display(Markdown("\n".join(lineas)))
res.registrar("global.conclusiones", lineas)
tiempos.reporte()